In [6]:
# ============================================================
# CORRECTED API FUNCTIONS
# Geodata + Climate data + Median by Admin 2 and Admin 1
# ============================================================

import requests
import pandas as pd
import geopandas as gpd
from tqdm import tqdm


# Correct API base URLs
GEODATA_API_URI_BASE = "https://api.vam.wfp.org/geodata/"
CLIMATE_API_URI_BASE = "https://api.earthobservation.vam.wfp.org/stats/"


def get_geodata(adm0: int, admin2: bool = False) -> gpd.GeoDataFrame:
    """
    Creates a GeoDataFrame with WFP administrative geometries.

    Parameters
    ----------
    adm0 : int
        Country code as set by WFP.
        Examples: 79 = Ethiopia, 115 = India, 182 = Nigeria, 269 = Yemen.
    admin2 : bool
        If False, returns Admin 1 only.
        If True, returns Admin 1 + Admin 2 data.

    Returns
    -------
    gpd.GeoDataFrame
    """

    def fetch_geojson(url: str):
        req = requests.get(url, timeout=60)
        req.raise_for_status()
        return req.json()

    # --------------------------------------------------------
    # Admin 1 geometries
    # --------------------------------------------------------
    adm1_url = GEODATA_API_URI_BASE + f"GetGeoAdmins?adm0={adm0}&admcode={adm0}"
    adm1_json = fetch_geojson(adm1_url)

    adm1_geo_df = (
        gpd.GeoDataFrame.from_features(adm1_json["features"])
        .rename(
            columns={
                "Name": "adm1Name",
                "Code": "adm1Code",
                "geometry": "adm1Geometry"
            }
        )
    )

    adm1_geo_df = gpd.GeoDataFrame(adm1_geo_df, geometry="adm1Geometry")

    if not admin2:
        return adm1_geo_df

    # --------------------------------------------------------
    # Admin 2 geometries
    # --------------------------------------------------------
    adm2_url = GEODATA_API_URI_BASE + f"Admin2ByCountry?adm0code={adm0}"
    adm2_json = fetch_geojson(adm2_url)

    adm2_geo_df = (
        gpd.GeoDataFrame.from_features(adm2_json["features"])
        .rename(
            columns={
                "Name": "adm2Name",
                "Code": "adm2Code",
                "geometry": "adm2Geometry"
            }
        )
    )

    adm2_geo_df = gpd.GeoDataFrame(adm2_geo_df, geometry="adm2Geometry")

    # --------------------------------------------------------
    # Admin 1 - Admin 2 relationship table
    # --------------------------------------------------------
    relations_url = GEODATA_API_URI_BASE + f"CountriesAndSubdivisions?adm0code={adm0}"
    relations_json = fetch_geojson(relations_url)

    relations_data = relations_json[0]["administrativeSubdivisions"]

    adm1_adm2 = (
        pd.json_normalize(
            relations_data,
            "administrativeSubdivisions",
            ["name"],
            meta_prefix="adm1_"
        )
        .rename(
            columns={
                "name": "adm2Name_relation",
                "adm1_name": "adm1Name"
            }
        )
    )

    adm1_adm2 = adm1_adm2.drop(columns=["localName"], errors="ignore")

    # Keep only needed relation columns
    relation_cols = ["adm1Name", "adm2Code"]
    if "adm2Name_relation" in adm1_adm2.columns:
        relation_cols.append("adm2Name_relation")

    adm1_adm2 = adm1_adm2[relation_cols].copy()

    # --------------------------------------------------------
    # Merge cleanly, avoiding adm2Name_x / adm2Name_y
    # --------------------------------------------------------
    geo = (
        adm1_adm2
        .merge(
            adm1_geo_df[["adm1Name", "adm1Code", "adm1Geometry"]],
            on="adm1Name",
            how="left"
        )
        .merge(
            adm2_geo_df[["adm2Code", "adm2Name", "adm2Geometry"]],
            on="adm2Code",
            how="left"
        )
    )

    # If adm2Name is missing from geometry, use relation name
    if "adm2Name" not in geo.columns and "adm2Name_relation" in geo.columns:
        geo["adm2Name"] = geo["adm2Name_relation"]

    if "adm2Name" in geo.columns and "adm2Name_relation" in geo.columns:
        geo["adm2Name"] = geo["adm2Name"].fillna(geo["adm2Name_relation"])

    geo = geo.drop(columns=["adm2Name_relation"], errors="ignore")

    # Reorder columns
    ordered_cols = [
        "adm1Name",
        "adm1Code",
        "adm2Name",
        "adm2Code",
        "adm1Geometry",
        "adm2Geometry"
    ]

    geo = geo[[c for c in ordered_cols if c in geo.columns]]

    geo = gpd.GeoDataFrame(geo, geometry="adm2Geometry")

    return geo


def get_climate_data(
    all_adm2: gpd.GeoDataFrame,
    start_date: str = "2017-01-01",
    end_date: str = "2019-12-21",
    cols: list = ["rfh_avg", "r3q", "vim_avg"],
    coverage: str = "full",
    level: int = 2,
    timeout: int = 60,
    verbose: bool = True
):
    """
    Extracts climate data from the corrected WFP Earth Observation API.

    Parameters
    ----------
    all_adm2 : gpd.GeoDataFrame
        Admin 2 + Admin 1 dataframe from get_geodata(..., admin2=True).
    start_date : str
        Start date in yyyy-mm-dd format.
    end_date : str
        End date in yyyy-mm-dd format.
    cols : list
        Climate variables to retain.
    coverage : str
        Usually 'full', 'past', or 'crop'.
    level : int
        Administrative level. Usually 2 for Admin 2.
    timeout : int
        Request timeout in seconds.
    verbose : bool
        If True, prints progress and failures.

    Returns
    -------
    df_climate : pd.DataFrame
        Climate data merged with admin names/codes.
    failed_requests : pd.DataFrame
        Log of failed or empty API calls.
    """

    climate_data = []
    failed_requests = []

    # Ensure expected name column exists
    all_adm2 = all_adm2.copy()

    if "adm2Name" not in all_adm2.columns:
        raise KeyError(
            "all_adm2 must contain 'adm2Name'. "
            "Run the corrected get_geodata(..., admin2=True) first."
        )

    if "adm2Code" not in all_adm2.columns:
        raise KeyError("all_adm2 must contain 'adm2Code'.")

    iterator = all_adm2.dropna(subset=["adm2Code"]).drop_duplicates("adm2Code")

    for _, row in tqdm(iterator.iterrows(), total=len(iterator)):
        code = int(row["adm2Code"])

        url = (
            f"{CLIMATE_API_URI_BASE}admin?"
            f"id_code={code}&level={level}&coverage={coverage}&vam=all"
            f"&start={start_date}&end={end_date}&format=json"
        )

        try:
            response = requests.get(url, timeout=timeout)

            if response.status_code != 200:
                failed_requests.append(
                    {
                        "adm2Code": code,
                        "status_code": response.status_code,
                        "url": url,
                        "response_start": response.text[:300]
                    }
                )

                if verbose:
                    print(f"API request failed for adm2Code {code}: status {response.status_code}")

                continue

            data = response.json()

            if not data or not data.get("data"):
                failed_requests.append(
                    {
                        "adm2Code": code,
                        "status_code": "no_data",
                        "url": url,
                        "response_start": "No data field"
                    }
                )
                continue

            # Some API responses may contain all-None variables
            if all(value is None for value in data["data"].values()):
                failed_requests.append(
                    {
                        "adm2Code": code,
                        "status_code": "all_none",
                        "url": url,
                        "response_start": "All data values are None"
                    }
                )
                continue

            df_temp = pd.DataFrame(data["data"])

            if "date" in data:
                df_temp["dates"] = data["date"]
            elif "dates" in data:
                df_temp["dates"] = data["dates"]
            else:
                df_temp["dates"] = pd.NA

            df_temp["adm2Code"] = code

            climate_data.append(df_temp)

        except requests.Timeout:
            failed_requests.append(
                {
                    "adm2Code": code,
                    "status_code": "timeout",
                    "url": url,
                    "response_start": "Request timed out"
                }
            )

            if verbose:
                print(f"Request timed out for adm2Code {code}")

        except requests.RequestException as e:
            failed_requests.append(
                {
                    "adm2Code": code,
                    "status_code": "request_error",
                    "url": url,
                    "response_start": str(e)[:300]
                }
            )

            if verbose:
                print(f"API request failed for adm2Code {code}: {e}")

    failed_requests = pd.DataFrame(failed_requests)

    if not climate_data:
        empty_cols = cols + ["dates", "adm2Code"]
        df_empty = pd.DataFrame(columns=empty_cols)
        df_empty = df_empty.merge(all_adm2, on="adm2Code", how="right")
        return df_empty, failed_requests

    df_climate = pd.concat(climate_data, ignore_index=True)

    # Merge with clean admin metadata
    metadata_cols = ["adm1Name", "adm1Code", "adm2Name", "adm2Code"]

    metadata = (
        all_adm2[metadata_cols]
        .drop_duplicates("adm2Code")
        .copy()
    )

    df_climate = df_climate.merge(metadata, on="adm2Code", how="left")

    # Convert selected climate columns to numeric
    for col in cols:
        if col in df_climate.columns:
            df_climate[col] = pd.to_numeric(df_climate[col], errors="coerce")
        else:
            df_climate[col] = pd.NA

    return df_climate, failed_requests


def calculate_climate_medians(
    df_climate: pd.DataFrame,
    climate_cols: list = ["rfh_avg", "r3q", "vim_avg"],
    start_date: str | None = None,
    end_date: str | None = None
):
    """
    Calculates median climate values by Admin 2 and Admin 1.

    Parameters
    ----------
    df_climate : pd.DataFrame
        Output from get_climate_data().
    climate_cols : list
        Climate variables to aggregate.
    start_date : str or None
        Optional start date to add to output.
    end_date : str or None
        Optional end date to add to output.

    Returns
    -------
    median_admin2 : pd.DataFrame
    median_admin1 : pd.DataFrame
    """

    required_cols = ["adm1Name", "adm1Code", "adm2Name", "adm2Code"] + climate_cols

    missing = [c for c in required_cols if c not in df_climate.columns]

    if missing:
        print("Missing columns:", missing)
        print("Available columns:")
        print(df_climate.columns.tolist())
        raise KeyError(f"Missing required columns: {missing}")

    df = df_climate[required_cols].copy()

    for col in climate_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    median_admin2 = (
        df
        .groupby(["adm1Name", "adm1Code", "adm2Name", "adm2Code"], as_index=False)[climate_cols]
        .median()
    )

    median_admin1 = (
        df
        .groupby(["adm1Name", "adm1Code"], as_index=False)[climate_cols]
        .median()
    )

    if start_date is not None:
        median_admin2["start_date"] = start_date
        median_admin1["start_date"] = start_date

    if end_date is not None:
        median_admin2["end_date"] = end_date
        median_admin1["end_date"] = end_date

    return median_admin2, median_admin1

In [7]:
# ============================================================
# Create a minimal API object to replace "self"
# ============================================================

class APIConfig:
    GEODATA_API_URI_BASE = "https://api.vam.wfp.org/geodata/"
    CLIMATE_API_URI_BASE = "https://api.earthobservation.vam.wfp.org/stats/"

api = APIConfig()

In [8]:
country_code = 170  # Nigeria = 182, Yemen = 269, Zimbabwe =271, moz=170
start_date = "2021-01-01"
end_date = "2023-12-31"

climate_cols = ["rfh_avg", "r3q", "vim_avg"]

# 1. Get Admin 1 + Admin 2 geometries
all_adm2 = get_geodata(
    adm0=country_code,
    admin2=True
)

print("Admin-2 geometries loaded:")
print(all_adm2.shape)
display(all_adm2.head())


# 2. Download climate data
df_climate, failed_requests = get_climate_data(
    all_adm2=all_adm2,
    start_date=start_date,
    end_date=end_date,
    cols=climate_cols,
    coverage="full",
    level=2,
    timeout=60,
    verbose=True
)

Admin-2 geometries loaded:
(158, 6)


,adm1Name,adm1Code,adm2Name,adm2Code,adm1Geometry,adm2Geometry
0,Cabo_Delgado,900948,Ancuabe,1010505,"MULTIPOLYGON (((40.5 -10.5699, 40.5148 -10.566...","POLYGON ((40.0275 -12.8389, 40.0257 -12.8178, ..."
1,Cabo_Delgado,900948,Balama,1010508,"MULTIPOLYGON (((40.5 -10.5699, 40.5148 -10.566...","POLYGON ((38.7073 -13.4828, 38.6742 -13.4611, ..."
2,Cabo_Delgado,900948,Chiure,1010525,"MULTIPOLYGON (((40.5 -10.5699, 40.5148 -10.566...","POLYGON ((39.324 -13.8925, 39.3335 -13.8933, 3..."
3,Cabo_Delgado,900948,Cidade_De_Pemba,1010536,"MULTIPOLYGON (((40.5 -10.5699, 40.5148 -10.566...","POLYGON ((40.4716 -13.1086, 40.4999 -13.085, 4..."
4,Cabo_Delgado,900948,Ibo,1010554,"MULTIPOLYGON (((40.5 -10.5699, 40.5148 -10.566...","MULTIPOLYGON (((40.6164 -12.4213, 40.596 -12.4..."


100%|██████████| 158/158 [00:15<00:00, 10.11it/s]


In [9]:
# ============================================================
# AGGREGATE CLIMATE DATA OVER THE WHOLE PERIOD
# One Admin-2-level GeoDataFrame with:
# - Admin 2 medians: rfh_avg_2, r3q_2, vim_avg_2
# - Admin 1 medians: rfh_avg_1, r3q_1, vim_avg_1
# - Admin 1 geometry
# - Admin 2 geometry
# ============================================================

import geopandas as gpd
import pandas as pd

climate_cols = ["rfh_avg", "r3q", "vim_avg"]

# ------------------------------------------------------------
# 1. Make sure dates and climate variables are correctly typed
# ------------------------------------------------------------

df_climate["dates"] = pd.to_datetime(df_climate["dates"], errors="coerce")

for col in climate_cols:
    df_climate[col] = pd.to_numeric(df_climate[col], errors="coerce")


# ------------------------------------------------------------
# 2. Median over the whole period by Admin 2
# ------------------------------------------------------------

median_admin2 = (
    df_climate
    .groupby(
        ["adm1Name", "adm1Code", "adm2Name", "adm2Code"],
        as_index=False
    )[climate_cols]
    .median()
)

median_admin2 = median_admin2.rename(
    columns={col: f"{col}_2" for col in climate_cols}
)


# ------------------------------------------------------------
# 3. Median over the whole period by Admin 1
# ------------------------------------------------------------

median_admin1 = (
    df_climate
    .groupby(
        ["adm1Name", "adm1Code"],
        as_index=False
    )[climate_cols]
    .median()
)

median_admin1 = median_admin1.rename(
    columns={col: f"{col}_1" for col in climate_cols}
)


# ------------------------------------------------------------
# 4. Merge Admin 1 medians onto Admin 2 rows
# ------------------------------------------------------------

climate_medians = median_admin2.merge(
    median_admin1,
    on=["adm1Name", "adm1Code"],
    how="left"
)


# ------------------------------------------------------------
# 5. Add Admin 1 and Admin 2 geometries from all_adm2
# ------------------------------------------------------------

geometry_cols = [
    "adm1Name",
    "adm1Code",
    "adm2Name",
    "adm2Code",
    "adm1Geometry",
    "adm2Geometry"
]

geometries = all_adm2[geometry_cols].drop_duplicates("adm2Code").copy()

climate_medians_geo = climate_medians.merge(
    geometries,
    on=["adm1Name", "adm1Code", "adm2Name", "adm2Code"],
    how="left"
)


# ------------------------------------------------------------
# 7. Convert to GeoDataFrame using Admin 2 geometry as active geometry
# ------------------------------------------------------------

climate_medians_geo = gpd.GeoDataFrame(
    climate_medians_geo,
    geometry="adm2Geometry",
    crs=all_adm2.crs
)

# Important:
# GeoPandas allows one active geometry column.
# adm2Geometry is the active geometry.
# adm1Geometry is kept as an additional geometry-like column.


# ------------------------------------------------------------
# 8. Inspect final output
# ------------------------------------------------------------

print("Final GeoDataFrame:")
print(climate_medians_geo.shape)

display(climate_medians_geo.head())

print("Columns:")
print(climate_medians_geo.columns.tolist())


# ------------------------------------------------------------
# 9. Save as GeoPackage
# ------------------------------------------------------------

climate_medians_geo.to_csv("climate_medians.csv", index=False)

print("File saved:")
print("climate_medians.csv")

Final GeoDataFrame:
(158, 12)


,adm1Name,adm1Code,adm2Name,adm2Code,rfh_avg_2,r3q_2,vim_avg_2,rfh_avg_1,r3q_1,vim_avg_1,adm1Geometry,adm2Geometry
0,Cabo_Delgado,900948,Ancuabe,1010505,6.71145,92.79795,0.63685,7.9858,98.75785,0.65605,"MULTIPOLYGON (((40.5 -10.5699, 40.5148 -10.566...","POLYGON ((40.0275 -12.8389, 40.0257 -12.8178, ..."
1,Cabo_Delgado,900948,Balama,1010508,6.52025,94.44390,0.61640,7.9858,98.75785,0.65605,"MULTIPOLYGON (((40.5 -10.5699, 40.5148 -10.566...","POLYGON ((38.7073 -13.4828, 38.6742 -13.4611, ..."
2,Cabo_Delgado,900948,Chiure,1010525,6.02005,94.11935,0.60865,7.9858,98.75785,0.65605,"MULTIPOLYGON (((40.5 -10.5699, 40.5148 -10.566...","POLYGON ((39.324 -13.8925, 39.3335 -13.8933, 3..."
3,Cabo_Delgado,900948,Cidade_De_Pemba,1010536,8.01115,99.81410,0.39410,7.9858,98.75785,0.65605,"MULTIPOLYGON (((40.5 -10.5699, 40.5148 -10.566...","POLYGON ((40.4716 -13.1086, 40.4999 -13.085, 4..."
4,Cabo_Delgado,900948,Ibo,1010554,8.68335,111.27890,0.74125,7.9858,98.75785,0.65605,"MULTIPOLYGON (((40.5 -10.5699, 40.5148 -10.566...","MULTIPOLYGON (((40.6164 -12.4213, 40.596 -12.4..."


Columns:
['adm1Name', 'adm1Code', 'adm2Name', 'adm2Code', 'rfh_avg_2', 'r3q_2', 'vim_avg_2', 'rfh_avg_1', 'r3q_1', 'vim_avg_1', 'adm1Geometry', 'adm2Geometry']
File saved:
climate_medians.csv
